In [5]:
!pip install openai

from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("請輸入你的OpenAI API key：")

In [15]:
from openai import OpenAI

client = OpenAI()

system_prompt = """你是一位負責辦公大樓能源管理的節電顧問。你會收到一棟辦公建築的用電異常分析結果，並針對異常類型提供對應的節能建議。

【重要限制，務必遵守】
1. 你只會拿到「整棟建築」的電表資料，沒有空調、照明、電腦等individual設備的分項用電量。絕對不可以假裝知道或編造某個設備佔多少百分比的用電量（例如不可以說「空調佔用電量的33%」），這是資料本身沒有的資訊。
2. 你的建議必須是「建築層級」或「系統層級」的措施，例如：空調系統排程調整、建築外殼隔熱、公共區域照明時間設定、尖峰時段設備輪流啟動、樓層分區空調管理等，而不是針對單一設備的操作建議。
3. 不要編造具體的節電百分比或金額，除非是根據下方參考表中的數字，且必須註明「一般而言」、「業界案例顯示」、「文獻案例顯示」等字樣，不可以宣稱是「根據您的用電數據算出」。
4. 根據異常類型，你的建議行動層級要不同：
   - 「預測區間超出」異常 → 屬於即時性事件，建議語氣為「近期檢查」，內容偏向立即可執行的短期措施
   - 「時段違常」異常 → 屬於排程問題，建議語氣為「排程調整」，內容偏向作息時間、設備啟閉排程
   - 「同儕落後」異常 → 屬於結構性長期落後，建議語氣為「長期評估」，內容偏向設備汰換、外殼改善、系統性節能規劃

【可引用的量化參考（僅能使用此表內數字與出處，不可自行編造其他百分比或案例）】
1. 空調溫度設定調整：溫度每提高1℃，約可降低6%空調用電（台電節能建議）
2. 空調系統變頻/時段控制：業界案例顯示，導入變頻(VRF)空調系統並搭配時段控制，年省電費約30%（台中市冷凍空調技師公會／台電雲林區營業處案例分享）
3. 能源管理系統(EMS)導入或優化：(a) 文獻案例顯示平均約4.1%/年整體用電節省（Fitzgerald et al., 2023，ISO 50001案例研究）；(b) 業界案例顯示，使用超過10年的建築導入能源管理機制，至少可省15%用電量（森威能源節能案例分享，引用於CSR@天下，2021）
4. 建築外殼隔熱改善（特定技術措施，優先使用此項而非第5項的法規統計）：業界案例顯示，特殊降溫塗料搭配灑水系統可使建築空調負載降低30-40%（政府委託研究案例）
5. 建築節能法規長期效果（全國政策層級統計，僅在談論法規/政策脈絡時使用，不要用於單一建築的改善行動建議效果）：約可降低16%建築空調尖峰用電量（內政部建築研究所相關研究，實施20年後估算）
6. LED/高效照明汰換：約可降低60-80%照明用電（經濟部能源署節能技術手冊）

規則：
- 每一個引用數字後面「必須」立即用括號標明出處名稱，格式為「XX%（出處名稱）」，不可只寫數字不寫出處。
- 若同一則建議引用超過一項參考表數字，每一項都要各自附上自己的括號出處，用分號隔開。
- 若建議內容不屬於上述六類措施，則不附加任何量化數字，只描述措施內容本身。
- 針對單一建築的「外殼隔熱改善」行動建議，請優先引用第4項（30-40%特定技術案例），不要引用第5項（16%是全國法規20年後的總體統計，不是單一建築改善行動的預期效果）。

【何時可以引用參考效益數字，何時不行——請嚴格依此判斷，不可一律套用同一種寫法】
- 「時段違常」異常，依超出基線比例分三種情況處理，語氣與是否量化都不同：
  (a) 超出比例極小、接近0%（例如0%~1%左右）：屬於邊界情形，不引用任何數字，說明「本案數值屬邊界情形，暫不提供量化節電效益估計」。
  (b) 超出比例明顯但未達故障等級（大約1%以上、未達300%）：應引用參考表第1或2項數字。
  (c) 超出比例極端（約300%以上，像設備故障）：不引用任何數字，說明「本案偏差幅度過大，已非一般排程優化可完全解釋，暫不提供量化效益估計，應以查明根本原因為優先」。
- 「同儕落後」異常：百分位越高，代表落後越嚴重，引用的參考表項目數量也應該越多，且用詞要反映嚴重程度差異：
  (a) 百分位0.75~0.80左右（剛達到落後門檻）：只引用1項，建議用EMS(4.1%/年，Fitzgerald et al., 2023)作為起步評估建議。
  (b) 百分位0.80~0.90左右（明顯落後）：引用2項，例如EMS加上外殼隔熱第4項(30-40%)。
  (c) 百分位接近1.00（同儕群組中最差）：引用3項，包含EMS、外殼隔熱第4項、以及LED第6項(60-80%)，反映需要最全面性的長期改善。
  不要對不同百分位的案例使用完全相同的引用組合，引用的項目數量與內容要跟著百分位變化。

【輸出格式，請嚴格依照此三段式結構】
1. 異常診斷說明（1-2句話，說明是什麼類型的異常、大致嚴重程度）
2. 建議行動（2-4項，依異常類型調整行動層級）
3. 參考效益（依照上方規則判斷後撰寫，不要不分程度一律使用同一句話或同一組引用）

【範例一：時段違常類，超出基線0%的邊界案例——不引用任何數字】
1. 異常診斷說明：本次離峰時段用電強度與該建築離峰基準值幾乎相同，屬於邊界性案例，尚不足以判定為明確的排程異常，建議列為觀察對象即可。
2. 建議行動：（1）持續監測該時段用電是否穩定貼近基準值，若後續多次出現於基準值以上，再進一步排查；（2）確認離峰時段空調與照明設備是否依排定時間關閉。
3. 參考效益：本案數值屬邊界情形，暫不提供量化節電效益估計。

【範例二：時段違常類，超出基線僅6%的溫和案例——必須引用數字】
1. 異常診斷說明：本次離峰時段（下班後時段）用電強度超出該建築離峰基準值約6%，顯示下班後設備關閉時間可能有延遲。
2. 建議行動：（1）排程調整：檢查空調恆溫器與照明的下班關閉排程設定，確認是否與實際下班時間一致；（2）近期檢查：確認是否有加班人員導致部分區域仍在運作，若是則屬正常情形；（3）評估導入分區時段控制，避免全樓層設備延遲關閉。
3. 參考效益：業界案例顯示，導入空調時段控制搭配變頻(VRF)系統，年可省電費約30%（台中市冷凍空調技師公會／台電雲林區營業處案例分享），可作為評估分區時段控制導入效益的參考。

【範例三：時段違常類，超出基線1462%的極端故障等級案例——不引用數字】
1. 異常診斷說明：本建築於離峰時段的用電強度為基線的15.62倍，偏差幅度極為顯著，已遠超一般排程延遲可解釋的範圍，需視為需立即處理的異常事件。
2. 建議行動：（1）近期檢查：立即確認是否有大型設備異常啟動、故障運轉或錯誤連動，排除設備故障可能性；（2）近期檢查：確認是否有非預期的人員進出或系統控制邏輯錯誤；（3）排程調整：待排除設備異常後，重新檢視並鎖定該時段的自動關閉排程。
3. 參考效益：本案偏差幅度過大，已非一般排程優化可完全解釋，暫不提供量化效益估計，應以查明根本原因為優先。

【範例四：同儕落後類，第75百分位（剛達門檻）——只引用1項】
1. 異常診斷說明：本建築EUI於同儕群組中恰位於第75百分位，屬於同儕群組中用電效率相對落後的邊界案例，尚未達到嚴重落後程度。
2. 建議行動：（1）長期評估：檢視建築外殼隔熱與空調系統效率是否有可提升空間；（2）長期評估：建議先以能源管理系統（EMS）盤點用電結構，作為後續改善的基礎資料。
3. 參考效益：文獻案例顯示，EMS導入平均約可帶來4.1%/年整體用電節省（Fitzgerald et al., 2023，ISO 50001案例研究）。

【範例五：同儕落後類，第100百分位（同儕群組最差）——引用3項】
1. 異常診斷說明：本建築EUI於同儕群組中位居第100百分位，即該群組中用電效率最落後者，屬於結構性節能落後最嚴重的案例。
2. 建議行動：（1）長期評估：全面性建築能源診斷，涵蓋外殼隔熱、空調系統效率、照明系統現況；（2）長期評估：導入EMS並設定分項監控，建立長期改善追蹤基礎；（3）長期評估：研擬分階段設備汰換計畫；（4）長期評估：檢視照明系統是否仍有傳統燈具可汰換為高效光源。
3. 參考效益：業界案例顯示，特殊降溫塗料搭配灑水系統可使建築空調負載降低30-40%（政府委託研究案例）；文獻案例顯示，EMS導入平均約4.1%/年整體節省（Fitzgerald et al., 2023）；經濟部能源署節能技術手冊指出，全面替換LED照明約可降低60-80%照明用電。
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)

1. 異常診斷說明：本建築EUI於同儕群組中位居第100百分位，即該群組中用電效率最落後者，屬於結構性節能落後最嚴重的案例。

2. 建議行動：（1）長期評估：全面性建築能源診斷，涵蓋外殼隔熱、空調系統效率、照明系統現況；（2）長期評估：導入能源管理系統（EMS）並設定分項監控，建立長期改善追蹤基礎；（3）長期評估：研擬分階段設備汰換計畫；（4）長期評估：檢視照明系統是否仍有傳統燈具可汰換為高效光源。

3. 參考效益：業界案例顯示，特殊降溫塗料搭配灑水系統可使建築空調負載降低30-40%（政府委託研究案例）；文獻案例顯示，EMS導入平均約可帶來4.1%/年整體用電節省（Fitzgerald et al., 2023，ISO 50001案例研究）；經濟部能源署節能技術手冊指出，全面替換LED照明約可降低60-80%照明用電。


「排程調整」

In [16]:
user_prompt_rule2 = """【建築資訊】
建築代號：Hog_office_Terry
所在場站：Hog
建築樓地板面積：約3,200 平方公尺（示意值）
電表類型：整棟電力（electricity）

【異常分析結果】
異常類型：時段違常（Schedule Deviation）
時間：2017-09-09（週六）16:00
用電強度：0.176
該建築離峰時段基線：0.151
說明：本時段屬於離峰時間（假日），用電強度高於該建築自身離峰基線約17%。

請依照上述資訊，產生節能建議。"""

response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_rule2},
    ],
    temperature=0.3,
)

print(response2.choices[0].message.content)

1. 異常診斷說明：本建築於假日離峰時段的用電強度超出基線約17%，顯示該時段設備關閉時間可能有延遲或不當使用情形。

2. 建議行動：（1）排程調整：檢查空調與照明的假日關閉排程設定，確認是否與實際使用情況一致；（2）近期檢查：確認是否有加班人員或特殊活動導致部分區域仍在運作，若是則屬正常情形；（3）評估導入分區時段控制，避免全樓層設備在假日延遲關閉。

3. 參考效益：業界案例顯示，導入空調時段控制搭配變頻(VRF)系統，年可省電費約30%（台中市冷凍空調技師公會／台電雲林區營業處案例分享），可作為評估分區時段控制導入效益的參考。


In [17]:
cases = [
    # 規則二案例
    {"id":"R2-1","type":"schedule","building_id":"Bobcat_office_Alma","site_id":"Bobcat","sqm":1060.9,
     "timestamp":"2016-01-08 20:00","load_intensity":0.0218,"baseline":0.0218,"ratio":1.00},
    {"id":"R2-2","type":"schedule","building_id":"Rat_office_Colby","site_id":"Rat","sqm":79000.4,
     "timestamp":"2016-01-13 07:00","load_intensity":0.1104,"baseline":0.1082,"ratio":1.02},
    {"id":"R2-3","type":"schedule","building_id":"Hog_office_Cornell","site_id":"Hog","sqm":62385.0,
     "timestamp":"2016-04-21 17:00","load_intensity":0.1013,"baseline":0.0952,"ratio":1.06},
    {"id":"R2-4","type":"schedule","building_id":"Panther_office_Jesus","site_id":"Panther","sqm":9595.6,
     "timestamp":"2016-05-01 19:00","load_intensity":0.0775,"baseline":0.0672,"ratio":1.15},
    {"id":"R2-5","type":"schedule","building_id":"Bull_office_Sally","site_id":"Bull","sqm":7705.3,
     "timestamp":"2016-01-20 01:00","load_intensity":0.9183,"baseline":0.0588,"ratio":15.62},
    # 規則三案例
    {"id":"R3-1","type":"peer","building_id":"Bobcat_office_Nikita","site_id":"Bobcat","sqm":3895.7,
     "EUI":395.9,"percentile":0.750},
    {"id":"R3-2","type":"peer","building_id":"Eagle_office_Remedios","site_id":"Eagle","sqm":4781.8,
     "EUI":703.4,"percentile":0.800},
    {"id":"R3-3","type":"peer","building_id":"Eagle_office_Francis","site_id":"Eagle","sqm":14807.3,
     "EUI":949.9,"percentile":0.875},
    {"id":"R3-4","type":"peer","building_id":"Cockatoo_office_Laila","site_id":"Cockatoo","sqm":10254.6,
     "EUI":886.2,"percentile":1.000},
    {"id":"R3-5","type":"peer","building_id":"Wolf_office_Elisabeth","site_id":"Wolf","sqm":2591.0,
     "EUI":2133.7,"percentile":1.000},
]

def build_user_prompt(case):
    if case["type"] == "schedule":
        weekday_note = "假日" if case.get("is_weekend") else "平日"
        return f"""【建築資訊】
建築代號：{case['building_id']}
所在場站：{case['site_id']}
建築樓地板面積：{case['sqm']:.1f} 平方公尺
電表類型：整棟電力（electricity）

【異常分析結果】
異常類型：時段違常（Schedule Deviation）
時間：{case['timestamp']}
用電強度：{case['load_intensity']:.4f}
該建築離峰時段基線：{case['baseline']:.4f}
說明：本時段屬於離峰時間，用電強度高於該建築自身離峰基線約{(case['ratio']-1)*100:.0f}%。

請依照上述資訊，產生節能建議。"""
    else:
        return f"""【建築資訊】
建築代號：{case['building_id']}
所在場站：{case['site_id']}
建築樓地板面積：{case['sqm']:.1f} 平方公尺
電表類型：整棟電力（electricity）

【異常分析結果】
異常類型：同儕落後（Peer Benchmarking）
本建築用電強度指標（EUI）：{case['EUI']:.1f} kWh/m²（訓練期間累積）
百分位排名：位居同儕組第{case['percentile']*100:.1f}百分位
說明：本建築用電強度持續高於同規模同場站建築，非單一時間點異常，而是長期普遍偏高。

請依照上述資訊，產生節能建議。"""

results_v2 = []
for case in cases:
    user_prompt = build_user_prompt(case)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )
    generated = response.choices[0].message.content
    results_v2.append({**case, "user_prompt": user_prompt, "generated": generated})
    print(f"完成 {case['id']}")

import json
with open("llm_generated_cases_v2.json", "w", encoding="utf-8") as f:
    json.dump(results_v2, f, ensure_ascii=False, indent=2)

print("全部完成，已存成 llm_generated_cases_v2.json")

完成 R2-1
完成 R2-2
完成 R2-3
完成 R2-4
完成 R2-5
完成 R3-1
完成 R3-2
完成 R3-3
完成 R3-4
完成 R3-5
全部完成，已存成 llm_generated_cases_v2.json
